# Classical ML Deep Dive — Interview-Ready Implementations

This notebook covers classical ML algorithms the way real interviews test them — not just API calls, but understanding internals, math intuition, and trade-offs.

**Sections:**
1. Linear Models from Scratch
2. Decision Tree Internals
3. Ensemble Methods — Why They Work
4. SVM — The Margin Concept
5. 20 Interview Questions with Code Answers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing, make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, classification_report
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.svm import SVC
import xgboost as xgb

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
np.random.seed(42)
print('Libraries loaded.')

---
## Section 1 — Linear Models from Scratch

Linear models are the foundation of ML. Understanding gradient descent at the code level is a must for interviews.

### 1.1 Load California Housing Dataset

In [ ]:
housing = fetch_california_housing()
X_house = housing.data
y_house = housing.target

print('Shape:', X_house.shape)
print('Features:', housing.feature_names)
print('Target (median house value in $100k):', y_house[:5])

# Use 2 features for visualization clarity
X_lr = X_house[:, [0, 5]]  # MedInc, AveOccup
y_lr = y_house.copy()

# Standardize
scaler = StandardScaler()
X_lr_scaled = scaler.fit_transform(X_lr)

X_train, X_test, y_train, y_test = train_test_split(
    X_lr_scaled, y_lr, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### 1.2 Linear Regression — Pure Gradient Descent (No sklearn)

**Math:**
```
Prediction:  y_hat = X @ w + b
Loss (MSE):  L = (1/n) * sum((y - y_hat)^2)
Gradient dw: dL/dw = (-2/n) * X.T @ (y - y_hat)
Gradient db: dL/db = (-2/n) * sum(y - y_hat)
Update:      w = w - lr * dw
```

In [ ]:
class LinearRegressionGD:
    """Linear Regression via Gradient Descent."""
    def __init__(self, lr=0.01, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for i in range(self.n_iter):
            y_hat = X @ self.weights + self.bias
            error = y_hat - y

            dw = (2 / n_samples) * X.T @ error
            db = (2 / n_samples) * np.sum(error)

            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

            loss = np.mean(error ** 2)
            self.loss_history.append(loss)

        return self

    def predict(self, X):
        return X @ self.weights + self.bias


model_gd = LinearRegressionGD(lr=0.05, n_iter=2000)
model_gd.fit(X_train, y_train)

y_pred_gd = model_gd.predict(X_test)
print(f'Scratch GD  | MSE: {mean_squared_error(y_test, y_pred_gd):.4f}  R2: {r2_score(y_test, y_pred_gd):.4f}')

### 1.3 Visualize Loss Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(model_gd.loss_history, color='steelblue', lw=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Loss Convergence — Gradient Descent')
axes[0].set_yscale('log')

# Predicted vs Actual
axes[1].scatter(y_test[:200], y_pred_gd[:200], alpha=0.5, color='steelblue', s=20)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title('Predicted vs Actual (Scratch GD)')

plt.tight_layout()
plt.show()

### 1.4 Compare with sklearn — Should Match

In [ ]:
model_sk = LinearRegression()
model_sk.fit(X_train, y_train)
y_pred_sk = model_sk.predict(X_test)

print(f'Scratch GD  | MSE: {mean_squared_error(y_test, y_pred_gd):.4f}  R2: {r2_score(y_test, y_pred_gd):.4f}')
print(f'sklearn     | MSE: {mean_squared_error(y_test, y_pred_sk):.4f}  R2: {r2_score(y_test, y_pred_sk):.4f}')
print()
print('Weights from scratch GD:', np.round(model_gd.weights, 4))
print('Weights from sklearn:   ', np.round(model_sk.coef_, 4))
print()
print('Note: small differences expected — sklearn uses closed-form solution (normal equations).')

### 1.5 Logistic Regression from Scratch — Sigmoid + BCE Loss

**Math:**
```
Sigmoid:  sigma(z) = 1 / (1 + exp(-z))
BCE Loss: L = -(1/n) * sum(y*log(p) + (1-y)*log(1-p))
Gradient: dL/dw = (1/n) * X.T @ (p - y)
```

In [ ]:
class LogisticRegressionGD:
    """Logistic Regression via Gradient Descent."""
    def __init__(self, lr=0.1, n_iter=1000):
        self.lr = lr
        self.n_iter = n_iter
        self.weights = None
        self.bias = None
        self.loss_history = []

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        for _ in range(self.n_iter):
            z = X @ self.weights + self.bias
            p = self._sigmoid(z)

            dw = (1 / n_samples) * X.T @ (p - y)
            db = (1 / n_samples) * np.sum(p - y)

            self.weights -= self.lr * dw
            self.bias    -= self.lr * db

            eps = 1e-9
            loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
            self.loss_history.append(loss)

        return self

    def predict_proba(self, X):
        return self._sigmoid(X @ self.weights + self.bias)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


# Binary dataset from make_classification
X_cls, y_cls = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, random_state=42
)
scaler_cls = StandardScaler()
X_cls_scaled = scaler_cls.fit_transform(X_cls)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls_scaled, y_cls, test_size=0.2, random_state=42
)

lr_scratch = LogisticRegressionGD(lr=0.1, n_iter=500)
lr_scratch.fit(Xc_train, yc_train)

lr_sk = LogisticRegression(max_iter=1000)
lr_sk.fit(Xc_train, yc_train)

print(f'Scratch LR  | Accuracy: {accuracy_score(yc_test, lr_scratch.predict(Xc_test)):.4f}')
print(f'sklearn LR  | Accuracy: {accuracy_score(yc_test, lr_sk.predict(Xc_test)):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(lr_scratch.loss_history, color='darkorange', lw=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Logistic Regression — Loss Convergence')

# Decision boundary on 2 features for visualization
X2, y2 = make_classification(n_samples=500, n_features=2, n_informative=2,
                              n_redundant=0, random_state=42)
X2s = StandardScaler().fit_transform(X2)
lr2 = LogisticRegressionGD(lr=0.1, n_iter=300)
lr2.fit(X2s, y2)

hh = 0.05
x_min, x_max = X2s[:, 0].min() - 1, X2s[:, 0].max() + 1
y_min, y_max = X2s[:, 1].min() - 1, X2s[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, hh), np.arange(y_min, y_max, hh))
Z = lr2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
axes[1].scatter(X2s[:, 0], X2s[:, 1], c=y2, cmap='RdYlBu', edgecolors='k', s=20)
axes[1].set_title('Logistic Regression Decision Boundary (Scratch)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

---
## Section 2 — Decision Tree Internals

Decision trees are interpretable, fast, and the building block for Random Forest and XGBoost. Knowing how splits work is critical.

### 2.1 Implement Decision Tree from Scratch (Gini Impurity)

**Gini Impurity:**
```
Gini(S) = 1 - sum(p_i^2)   for each class i
Split quality = weighted_gini_left + weighted_gini_right
```

In [ ]:
class DecisionNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # leaf label


class DecisionTreeScratch:
    """Binary Decision Tree using Gini impurity."""
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    @staticmethod
    def _gini(y):
        n = len(y)
        if n == 0:
            return 0.0
        counts = np.bincount(y)
        probs = counts / n
        return 1 - np.sum(probs ** 2)

    def _best_split(self, X, y):
        best_gini = float('inf')
        best_feat, best_thresh = None, None
        n = len(y)

        for feat in range(X.shape[1]):
            thresholds = np.unique(X[:, feat])
            for thresh in thresholds:
                left_mask = X[:, feat] <= thresh
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                g = (left_mask.sum() / n) * self._gini(y[left_mask]) + \
                    (right_mask.sum() / n) * self._gini(y[right_mask])
                if g < best_gini:
                    best_gini = g
                    best_feat = feat
                    best_thresh = thresh
        return best_feat, best_thresh

    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split or len(np.unique(y)) == 1:
            return DecisionNode(value=np.bincount(y).argmax())

        feat, thresh = self._best_split(X, y)
        if feat is None:
            return DecisionNode(value=np.bincount(y).argmax())

        left_mask = X[:, feat] <= thresh
        left  = self._build(X[left_mask],  y[left_mask],  depth + 1)
        right = self._build(X[~left_mask], y[~left_mask], depth + 1)
        return DecisionNode(feature=feat, threshold=thresh, left=left, right=right)

    def fit(self, X, y):
        self.root = self._build(X, y, 0)
        return self

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])


print('DecisionTreeScratch defined.')

### 2.2 Load Titanic Dataset and Test Scratch Tree

In [ ]:
titanic = sns.load_dataset('titanic')
print(titanic.shape)
print(titanic.head(3))
print('\nMissing values:\n', titanic.isnull().sum())

In [ ]:
# Preprocess Titanic
titanic_clean = titanic[['survived','pclass','sex','age','sibsp','parch','fare']].copy()
titanic_clean['sex'] = (titanic_clean['sex'] == 'male').astype(int)
titanic_clean['age'].fillna(titanic_clean['age'].median(), inplace=True)
titanic_clean.dropna(inplace=True)

Xt = titanic_clean.drop('survived', axis=1).values
yt = titanic_clean['survived'].values

Xt_train, Xt_test, yt_train, yt_test = train_test_split(Xt, yt, test_size=0.2, random_state=42)

# Scratch tree — small depth for speed
tree_scratch = DecisionTreeScratch(max_depth=4)
tree_scratch.fit(Xt_train, yt_train)
yt_pred_scratch = tree_scratch.predict(Xt_test)

# sklearn tree
tree_sk = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_sk.fit(Xt_train, yt_train)
yt_pred_sk = tree_sk.predict(Xt_test)

print(f'Scratch DT (depth=4) | Accuracy: {accuracy_score(yt_test, yt_pred_scratch):.4f}')
print(f'sklearn  DT (depth=4) | Accuracy: {accuracy_score(yt_test, yt_pred_sk):.4f}')

### 2.3 Decision Boundary Visualization

In [ ]:
# 2-feature dataset for clear boundary visualization
X_db, y_db = make_classification(n_samples=300, n_features=2, n_informative=2,
                                  n_redundant=0, n_clusters_per_class=1, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
depths = [1, 3, 20]

for ax, depth in zip(axes, depths):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_db, y_db)

    hh = 0.05
    x_min, x_max = X_db[:, 0].min() - 0.5, X_db[:, 0].max() + 0.5
    y_min, y_max = X_db[:, 1].min() - 0.5, X_db[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, hh), np.arange(y_min, y_max, hh))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X_db[:, 0], X_db[:, 1], c=y_db, cmap='RdYlBu', edgecolors='k', s=20)
    train_acc = clf.score(X_db, y_db)
    ax.set_title(f'Depth={depth}  |  Train Acc={train_acc:.2f}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Decision Tree — Overfitting as Depth Increases', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Overfitting Demonstration — Train vs Test Accuracy

In [ ]:
depths = range(1, 25)
train_accs = []
test_accs  = []

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_db, y_db, test_size=0.3, random_state=42)

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(Xd_tr, yd_tr)
    train_accs.append(clf.score(Xd_tr, yd_tr))
    test_accs.append(clf.score(Xd_te, yd_te))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, 'b-o', label='Train Accuracy', markersize=5)
plt.plot(depths, test_accs,  'r-o', label='Test Accuracy',  markersize=5)
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Overfitting: Decision Tree Depth vs Accuracy')
plt.legend()
plt.axvline(x=depths[np.argmax(test_accs)], color='green', linestyle='--',
            label=f'Best depth = {depths[np.argmax(test_accs)]}')
plt.legend()
plt.show()

print(f'Best test accuracy at depth {depths[np.argmax(test_accs)]}: {max(test_accs):.4f}')

### 2.5 Visualize sklearn Decision Tree Structure

In [ ]:
feat_names = ['pclass','sex','age','sibsp','parch','fare']
tree_viz = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_viz.fit(Xt_train, yt_train)

plt.figure(figsize=(20, 8))
plot_tree(tree_viz, feature_names=feat_names, class_names=['Died','Survived'],
          filled=True, rounded=True, fontsize=10)
plt.title('Titanic Decision Tree (depth=3)')
plt.show()

---
## Section 3 — Ensemble Methods: Why They Work

Ensemble methods consistently win Kaggle competitions and dominate tabular ML benchmarks. Understanding *why* requires knowing bias-variance decomposition.

### 3.1 Bagging Intuition — Math Proof

**Key insight:** If we average N independent, identically distributed estimators each with variance sigma^2:
```
Var(mean) = sigma^2 / N
```
Variance shrinks linearly with the number of models!

**In practice:** Trees are correlated, so variance reduction is less than 1/N — but still substantial.

In [ ]:
# Simulation: Show variance reduction via bagging
np.random.seed(42)

X_bag, y_bag = make_classification(n_samples=500, n_features=10, n_informative=5,
                                    n_redundant=2, random_state=42)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_bag, y_bag, test_size=0.3, random_state=42)

n_estimators_list = [1, 5, 10, 20, 50, 100, 200]
repetitions = 15

mean_accs  = []
std_accs   = []

for n_est in n_estimators_list:
    accs = []
    for rep in range(repetitions):
        bag = BaggingClassifier(n_estimators=n_est, random_state=rep)
        bag.fit(Xb_tr, yb_tr)
        accs.append(bag.score(Xb_te, yb_te))
    mean_accs.append(np.mean(accs))
    std_accs.append(np.std(accs))

plt.figure(figsize=(10, 5))
plt.errorbar(n_estimators_list, mean_accs, yerr=std_accs,
             fmt='-o', capsize=5, color='steelblue', ecolor='gray', lw=2)
plt.xlabel('Number of Estimators')
plt.ylabel('Test Accuracy')
plt.title('Bagging: More Estimators = Lower Variance (error bars = std over 15 runs)')
plt.xscale('log')
plt.show()

print('Std at  1 estimator :', round(std_accs[0], 4))
print('Std at 200 estimators:', round(std_accs[-1], 4))
print('Variance reduced by factor:', round(std_accs[0]**2 / std_accs[-1]**2, 1))

### 3.2 Random Forest — Feature Importance

In [ ]:
# Train on Titanic data
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(Xt_train, yt_train)

print(f'Random Forest Accuracy: {rf.score(Xt_test, yt_test):.4f}')

importances = rf.feature_importances_
feature_names = ['pclass','sex','age','sibsp','parch','fare']
idxs = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(importances)), importances[idxs], color='steelblue')
plt.xticks(range(len(importances)), [feature_names[i] for i in idxs])
plt.ylabel('Feature Importance (Mean Decrease Impurity)')
plt.title('Random Forest Feature Importance — Titanic')
plt.show()

print('Feature Importances:')
for i in idxs:
    print(f'  {feature_names[i]:10s}: {importances[i]:.4f}')

### 3.3 AdaBoost — Visualize Sample Weights Each Round

**AdaBoost key idea:** Misclassified samples get higher weight in the next round, forcing the next weak learner to focus on hard examples.

In [ ]:
from sklearn.tree import DecisionTreeClassifier as DTC

# Simple 2D dataset so we can visualize weights
X_ada, y_ada = make_classification(n_samples=100, n_features=2, n_informative=2,
                                    n_redundant=0, n_clusters_per_class=1,
                                    flip_y=0.1, random_state=7)

# Manually simulate AdaBoost weight updates
n = len(y_ada)
w = np.ones(n) / n  # uniform weights
weight_history = [w.copy()]
error_history  = []
alpha_history  = []

for round_i in range(8):
    stump = DTC(max_depth=1)
    stump.fit(X_ada, y_ada, sample_weight=w)
    preds = stump.predict(X_ada)

    incorrect = (preds != y_ada).astype(float)
    err = np.dot(w, incorrect) / w.sum()
    err = np.clip(err, 1e-10, 1 - 1e-10)

    alpha = 0.5 * np.log((1 - err) / err)
    w = w * np.exp(-alpha * (2 * (preds == y_ada).astype(float) - 1))
    w /= w.sum()

    weight_history.append(w.copy())
    error_history.append(err)
    alpha_history.append(alpha)

print('Round  | Error   | Alpha (weight of this learner)')
print('-' * 50)
for i, (e, a) in enumerate(zip(error_history, alpha_history)):
    print(f'  {i+1:2d}   | {e:.4f}  | {a:.4f}')

In [ ]:
# Visualize weight evolution for first 4 rounds
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, ax in enumerate(axes.flatten()):
    w_round = weight_history[idx]
    scatter = ax.scatter(X_ada[:, 0], X_ada[:, 1],
                         c=y_ada, cmap='RdYlBu',
                         s=w_round * n * 80 + 10,
                         edgecolors='k', linewidths=0.5, alpha=0.8)
    ax.set_title(f'Round {idx} — Sample Weights (size = weight)' if idx == 0
                 else f'After Round {idx} — High weight = hard to classify')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('AdaBoost: Sample Weights Evolve — Hard Examples Get More Weight', fontsize=13)
plt.tight_layout()
plt.show()

### 3.4 XGBoost vs Random Forest — Speed and Accuracy Comparison

In [ ]:
import time

# Use make_classification — larger dataset
X_cmp, y_cmp = make_classification(
    n_samples=5000, n_features=20, n_informative=10,
    n_redundant=5, random_state=42
)
Xcmp_tr, Xcmp_te, ycmp_tr, ycmp_te = train_test_split(X_cmp, y_cmp, test_size=0.2, random_state=42)

results = {}

# Random Forest
t0 = time.time()
rf_cmp = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_cmp.fit(Xcmp_tr, ycmp_tr)
results['Random Forest'] = {
    'train_time': time.time() - t0,
    'accuracy': accuracy_score(ycmp_te, rf_cmp.predict(Xcmp_te))
}

# XGBoost
t0 = time.time()
xgb_cmp = xgb.XGBClassifier(n_estimators=200, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_cmp.fit(Xcmp_tr, ycmp_tr)
results['XGBoost'] = {
    'train_time': time.time() - t0,
    'accuracy': accuracy_score(ycmp_te, xgb_cmp.predict(Xcmp_te))
}

print(f"{'Model':<20} {'Accuracy':>10} {'Train Time (s)':>15}")
print('-' * 47)
for model, m in results.items():
    print(f"{model:<20} {m['accuracy']:>10.4f} {m['train_time']:>15.3f}")

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
models = list(results.keys())

axes[0].bar(models, [results[m]['accuracy'] for m in models],
            color=['steelblue', 'darkorange'])
axes[0].set_ylim(0.85, 1.0)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Accuracy Comparison')
for i, m in enumerate(models):
    axes[0].text(i, results[m]['accuracy'] + 0.002,
                 f"{results[m]['accuracy']:.4f}", ha='center', fontsize=11)

axes[1].bar(models, [results[m]['train_time'] for m in models],
            color=['steelblue', 'darkorange'])
axes[1].set_ylabel('Train Time (seconds)')
axes[1].set_title('Training Speed Comparison')
for i, m in enumerate(models):
    axes[1].text(i, results[m]['train_time'] + 0.01,
                 f"{results[m]['train_time']:.2f}s", ha='center', fontsize=11)

plt.suptitle('XGBoost vs Random Forest (5000 samples, 200 estimators)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 4 — SVM: The Margin Concept

SVMs find the maximum-margin hyperplane separating classes. Understanding the margin geometry and the kernel trick is essential for interviews.

### 4.1 Hard Margin vs Soft Margin — C Parameter

**C parameter controls the trade-off:**
- **Small C** = wide margin, more misclassifications allowed (high bias, low variance)
- **Large C** = narrow margin, fewer misclassifications (low bias, high variance)

In [ ]:
# Create near-linearly separable data
np.random.seed(42)
n_pts = 80
X_svm = np.r_[np.random.randn(n_pts, 2) - [2, 2],
               np.random.randn(n_pts, 2) + [2, 2]]
y_svm = np.array([-1]*n_pts + [1]*n_pts)

# Add some noise points
noise_idx = np.random.choice(len(X_svm), 10, replace=False)
X_svm[noise_idx] += np.random.randn(10, 2) * 3

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
C_values = [0.01, 0.1, 1.0, 100.0]

for ax, C in zip(axes, C_values):
    clf = SVC(kernel='linear', C=C)
    clf.fit(X_svm, y_svm)

    x_min, x_max = X_svm[:, 0].min() - 1, X_svm[:, 0].max() + 1
    y_min, y_max = X_svm[:, 1].min() - 1, X_svm[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                          np.linspace(y_min, y_max, 200))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-1e5, -1, 0, 1, 1e5],
                colors=['#fee0d2','#fc8d59','#91bfdb','#4575b4'], alpha=0.4)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1],
               linestyles=['--', '-', '--'],
               colors=['red', 'black', 'blue'], linewidths=[1.5, 2, 1.5])

    # Highlight support vectors
    sv = clf.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=120, facecolors='none',
               edgecolors='gold', linewidths=2.5, zorder=5)

    ax.scatter(X_svm[:, 0], X_svm[:, 1], c=y_svm, cmap='RdBu',
               edgecolors='k', s=25, alpha=0.8)

    margin = 2 / np.linalg.norm(clf.coef_)
    misclass = np.sum(clf.predict(X_svm) != y_svm)
    ax.set_title(f'C={C}\nMargin={margin:.2f}  Misclass={misclass}\nSVs={len(sv)}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('SVM Hard vs Soft Margin: Effect of C Parameter\n(gold circles = support vectors)', fontsize=13)
plt.tight_layout()
plt.show()

### 4.2 Kernel Trick — RBF Kernel on Non-Linearly Separable Data

**The Kernel Trick:** Instead of explicitly mapping data to a higher-dimensional space, compute dot products in that space implicitly using a kernel function:

```
Linear:  K(x, z) = x . z
RBF:     K(x, z) = exp(-gamma * ||x - z||^2)
Poly:    K(x, z) = (x . z + r)^d
```

This allows SVMs to learn non-linear boundaries without explicit feature engineering.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

kernels = [
    ('linear', {'C': 1.0}),
    ('poly',   {'C': 1.0, 'degree': 3}),
    ('rbf',    {'C': 1.0, 'gamma': 'scale'})
]

for ax, (kernel, params) in zip(axes, kernels):
    clf = SVC(kernel=kernel, **params)
    clf.fit(X_moons, y_moons)

    hh = 0.03
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, hh),
                          np.arange(y_min, y_max, hh))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='RdYlBu',
               edgecolors='k', s=25)
    sv = clf.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=120, facecolors='none',
               edgecolors='gold', linewidths=2, zorder=5)

    acc = clf.score(X_moons, y_moons)
    ax.set_title(f'Kernel: {kernel.upper()}\nAccuracy: {acc:.2f}')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('SVM Kernels on Moon Dataset (gold circles = support vectors)', fontsize=13)
plt.tight_layout()
plt.show()

### 4.3 Support Vectors — Detailed Visualization

In [ ]:
# Clean linearly separable data to clearly show support vectors
np.random.seed(0)
X_sv = np.r_[np.random.randn(30, 2) - [2, 2],
              np.random.randn(30, 2) + [2, 2]]
y_sv = np.array([-1]*30 + [1]*30)

clf_sv = SVC(kernel='linear', C=1.0)
clf_sv.fit(X_sv, y_sv)

plt.figure(figsize=(8, 7))

x_min, x_max = X_sv[:, 0].min() - 1, X_sv[:, 0].max() + 1
y_min, y_max = X_sv[:, 1].min() - 1, X_sv[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
Z = clf_sv.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contour(xx, yy, Z, levels=[-1, 0, 1],
            linestyles=['--', '-', '--'],
            colors=['red', 'black', 'blue'],
            linewidths=[1.5, 2.5, 1.5])

# All points
plt.scatter(X_sv[:, 0], X_sv[:, 1], c=y_sv, cmap='bwr', alpha=0.6,
            edgecolors='k', s=40, zorder=4)

# Support vectors
sv = clf_sv.support_vectors_
plt.scatter(sv[:, 0], sv[:, 1], s=200, facecolors='none',
            edgecolors='gold', linewidths=3, zorder=5, label='Support Vectors')

margin = 2 / np.linalg.norm(clf_sv.coef_)
plt.title(f'SVM — Linear Kernel\nMargin = {margin:.3f}  |  {len(sv)} Support Vectors')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend(fontsize=11)

# Annotate margin
plt.annotate('Margin boundary (-1)', xy=(-4, -0.5), fontsize=9, color='red')
plt.annotate('Decision boundary (0)', xy=(-4, 0.5), fontsize=9, color='black')
plt.annotate('Margin boundary (+1)', xy=(-4, 1.8), fontsize=9, color='blue')

plt.show()

### 4.4 When to Use SVM vs Tree-Based Models

| Criterion | SVM | Random Forest / XGBoost |
|-----------|-----|------------------------|
| Dataset size | Small-medium (<10k) | Any size |
| Feature types | Numeric, normalized | Mixed, raw |
| Missing values | Requires imputation | RF handles natively |
| Interpretability | Low (esp. kernel) | Medium (feature importance) |
| Training speed | Slow (O(n^2-n^3)) | Fast, parallelizable |
| Non-linearity | Via kernel trick | Built-in (splits) |
| Hyperparameters | C, gamma, kernel | Many but robust defaults |

**Rule of thumb:**
- Use SVM when: high-dimensional sparse data (text), small datasets, clear margin
- Use tree ensembles when: tabular data, large datasets, need interpretability

---
## Section 5 — 20 Interview Questions with Code Answers

Each question includes: plain-English explanation + code demonstration.

### Q1: How does Random Forest handle missing values?

**Answer:** Standard scikit-learn Random Forest does NOT handle missing values natively — you must impute first. However, techniques like surrogate splits (used in older CART implementations) find alternative split variables when the primary split feature is missing. XGBoost and LightGBM handle missingness natively by learning the best direction to send NaN values.

In [ ]:
from sklearn.impute import SimpleImputer

# Demonstrate: RF fails with NaN, imputer fixes it
X_q1 = np.array([[1, 2], [3, np.nan], [np.nan, 5], [7, 8],
                  [2, 3], [4, 6], [1, np.nan], [9, 2]])
y_q1 = np.array([0, 1, 0, 1, 0, 1, 0, 1])

# Without imputation - will error
try:
    rf_test = RandomForestClassifier(n_estimators=5, random_state=42)
    rf_test.fit(X_q1, y_q1)
    print('RF fit succeeded (unexpected)')
except ValueError as e:
    print(f'RF without imputation: ValueError -> {e}')

# With imputation - works
imp = SimpleImputer(strategy='mean')
X_q1_imp = imp.fit_transform(X_q1)
rf_test.fit(X_q1_imp, y_q1)
print('RF with mean imputation: SUCCESS')
print('Imputed values:', np.round(X_q1_imp, 2))

# XGBoost handles NaN natively
xgb_test = xgb.XGBClassifier(n_estimators=5, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_test.fit(X_q1, y_q1)
print('XGBoost with NaN: SUCCESS (handles internally)')

### Q2: What happens when you increase max_depth in a Decision Tree?

**Answer:** Increasing max_depth allows the tree to learn finer and finer decision rules. Low depth = high bias (underfitting). High depth = high variance (overfitting). The tree memorizes training data and performs poorly on unseen data.

In [ ]:
X_q2, y_q2 = make_classification(n_samples=500, n_features=5,
                                   n_informative=3, random_state=1)
Xq2_tr, Xq2_te, yq2_tr, yq2_te = train_test_split(X_q2, y_q2, test_size=0.3, random_state=1)

depths = list(range(1, 21))
train_scores, test_scores = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(Xq2_tr, yq2_tr)
    train_scores.append(dt.score(Xq2_tr, yq2_tr))
    test_scores.append(dt.score(Xq2_te, yq2_te))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_scores, 'b-o', label='Train', markersize=5)
plt.plot(depths, test_scores,  'r-o', label='Test',  markersize=5)
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Q2: Effect of max_depth — Bias-Variance Trade-off')
plt.legend()
plt.axvline(depths[test_scores.index(max(test_scores))], color='green',
            linestyle='--', label='Optimal depth')
plt.legend()
plt.show()
print(f'Optimal max_depth: {depths[test_scores.index(max(test_scores))]}')

### Q3: Why does XGBoost outperform Random Forest on tabular data?

**Answer:**
1. **Boosting vs Bagging:** XGBoost corrects previous errors (boosting) whereas RF averages independent trees (bagging). Boosting has lower bias.
2. **Regularization:** XGBoost has L1/L2 regularization on leaf weights — RF doesn't.
3. **Second-order gradients:** XGBoost uses Newton's method (2nd derivatives), enabling faster convergence.
4. **Missing values:** Handled natively via learned split directions.
5. **Pruning:** XGBoost prunes trees based on gain thresholds.

However, RF is more robust to hyperparameters and faster to train.

In [ ]:
# Show that XGBoost improves iteratively (boosting adds value each round)
X_q3, y_q3 = make_classification(n_samples=2000, n_features=20,
                                   n_informative=10, n_redundant=5, random_state=42)
Xq3_tr, Xq3_te, yq3_tr, yq3_te = train_test_split(X_q3, y_q3, test_size=0.2, random_state=42)

# Track XGBoost accuracy as we add more rounds
xgb_model = xgb.XGBClassifier(n_estimators=200, learning_rate=0.1,
                                verbosity=0, use_label_encoder=False,
                                eval_metric='error', random_state=42)
xgb_model.fit(Xq3_tr, yq3_tr,
               eval_set=[(Xq3_te, yq3_te)],
               verbose=False)

results_xgb = xgb_model.evals_result()
error_vals  = results_xgb['validation_0']['error']
acc_vals    = [1 - e for e in error_vals]

# Compare final RF
rf_q3 = RandomForestClassifier(n_estimators=200, random_state=42)
rf_q3.fit(Xq3_tr, yq3_tr)
rf_acc = rf_q3.score(Xq3_te, yq3_te)

plt.figure(figsize=(10, 5))
plt.plot(acc_vals, color='darkorange', lw=2, label='XGBoost (cumulative rounds)')
plt.axhline(rf_acc, color='steelblue', linestyle='--', lw=2,
            label=f'Random Forest Final: {rf_acc:.4f}')
plt.xlabel('Number of Boosting Rounds')
plt.ylabel('Test Accuracy')
plt.title('Q3: XGBoost Progressive Improvement vs Random Forest')
plt.legend()
plt.show()
print(f'XGBoost final : {acc_vals[-1]:.4f}')
print(f'Random Forest : {rf_acc:.4f}')

### Q4: What is the kernel trick and why does it help?

**Answer:** Many datasets are not linearly separable in the original feature space. We could map them to a higher (or infinite) dimensional space where they ARE separable. But computing explicit feature maps is expensive.

The kernel trick exploits the fact that SVMs only need dot products between data points, not the explicit features. A kernel function K(x,z) computes the dot product in the transformed space WITHOUT materializing the transformation:

```
RBF kernel: K(x, z) = exp(-gamma * ||x-z||^2)
This corresponds to an infinite-dimensional feature space!
```

In [ ]:
# Demonstrate: circles dataset - not linearly separable
from sklearn.datasets import make_circles

X_circ, y_circ = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original 2D space
axes[0].scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='RdYlBu',
                edgecolors='k', s=30)
axes[0].set_title('Original 2D Space\n(Not linearly separable)')
axes[0].set_xlabel('x1')
axes[0].set_ylabel('x2')

# Manual 3D lift: add r^2 = x1^2 + x2^2 as 3rd feature
from mpl_toolkits.mplot3d import Axes3D  # noqa
r2 = X_circ[:, 0]**2 + X_circ[:, 1]**2
ax3d = fig.add_subplot(132, projection='3d')
ax3d.scatter(X_circ[:, 0], X_circ[:, 1], r2,
             c=y_circ, cmap='RdYlBu', edgecolors='k', s=20, alpha=0.7)
ax3d.set_title('Lifted to 3D: (x1, x2, x1^2+x2^2)\n(Now linearly separable!)')
ax3d.set_xlabel('x1')
ax3d.set_ylabel('x2')
ax3d.set_zlabel('r^2')

# SVM with RBF kernel
clf_circ = SVC(kernel='rbf', C=1.0, gamma='scale')
clf_circ.fit(X_circ, y_circ)
hh = 0.03
x_min, x_max = X_circ[:, 0].min()-0.3, X_circ[:, 0].max()+0.3
y_min, y_max = X_circ[:, 1].min()-0.3, X_circ[:, 1].max()+0.3
xx, yy = np.meshgrid(np.arange(x_min, x_max, hh), np.arange(y_min, y_max, hh))
Z = clf_circ.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[2].contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
axes[2].scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap='RdYlBu',
                edgecolors='k', s=30)
axes[2].set_title(f'SVM RBF Kernel Result\nAccuracy: {clf_circ.score(X_circ, y_circ):.3f}')
axes[2].set_xlabel('x1')
axes[2].set_ylabel('x2')

plt.suptitle('Q4: The Kernel Trick — No Explicit Feature Mapping Needed', fontsize=13)
plt.tight_layout()
plt.show()

### Q5: What is Gini Impurity vs Information Gain?

**Gini Impurity:**
```
Gini(S) = 1 - sum(p_i^2)
```
- Measures probability that a random element is misclassified
- Range: [0, 0.5] for binary, [0, 1-1/k] for k classes
- Faster to compute (no log)

**Information Gain (Entropy-based):**
```
Entropy(S) = -sum(p_i * log2(p_i))
Gain = Entropy(parent) - weighted_avg(Entropy(children))
```
- Range: [0, log2(k)]
- Slightly more expensive (log), tends to prefer splits with more values

**Interview:** Both usually give similar results. sklearn Decision Tree defaults to Gini. Use criterion='entropy' for a small chance of better performance on imbalanced data.

In [ ]:
# Visualize Gini vs Entropy as a function of class probability (binary case)
p = np.linspace(0.001, 0.999, 200)
gini    = 2 * p * (1 - p)              # Gini for binary: 1 - p^2 - (1-p)^2 = 2p(1-p)
entropy = -p * np.log2(p) - (1-p) * np.log2(1-p)

plt.figure(figsize=(9, 5))
plt.plot(p, gini,    label='Gini Impurity', lw=2, color='steelblue')
plt.plot(p, entropy/2, label='Entropy/2 (scaled)', lw=2, color='darkorange', linestyle='--')
plt.xlabel('Probability of class 1')
plt.ylabel('Impurity')
plt.title('Q5: Gini vs Entropy — Binary Classification')
plt.legend()
plt.axvline(0.5, color='gray', linestyle=':', alpha=0.7)
plt.text(0.51, 0.4, 'Max\nimpurity\nat p=0.5', fontsize=10, color='gray')
plt.show()

### Q6: What is regularization and why do we need it?

**Answer:** Regularization penalizes model complexity to prevent overfitting.

- **L2 (Ridge):** Adds `lambda * sum(w_i^2)` to loss. Shrinks weights toward zero, never exactly zero.
- **L1 (Lasso):** Adds `lambda * sum(|w_i|)` to loss. Can zero out weights — performs feature selection.
- **Elastic Net:** Combines L1 + L2.

**When alpha is 0:** No regularization = overfit. When alpha is large: underfit.

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Generate noisy data where regularization matters
np.random.seed(42)
X_reg = np.random.randn(100, 50)  # 50 features, 100 samples -> prone to overfitting
true_coef = np.zeros(50)
true_coef[:5] = [3, -2, 1.5, -1, 0.8]  # only 5 relevant features
y_reg = X_reg @ true_coef + np.random.randn(100) * 0.5
Xreg_tr, Xreg_te, yreg_tr, yreg_te = train_test_split(X_reg, y_reg, test_size=0.3, random_state=42)

alphas = np.logspace(-4, 3, 50)
ridge_test_mse = []
lasso_test_mse = []

for a in alphas:
    ridge = Ridge(alpha=a).fit(Xreg_tr, yreg_tr)
    lasso = Lasso(alpha=a, max_iter=5000).fit(Xreg_tr, yreg_tr)
    ridge_test_mse.append(mean_squared_error(yreg_te, ridge.predict(Xreg_te)))
    lasso_test_mse.append(mean_squared_error(yreg_te, lasso.predict(Xreg_te)))

plt.figure(figsize=(10, 5))
plt.semilogx(alphas, ridge_test_mse, 'b-o', markersize=4, label='Ridge Test MSE')
plt.semilogx(alphas, lasso_test_mse, 'r-o', markersize=4, label='Lasso Test MSE')
plt.xlabel('Alpha (regularization strength)')
plt.ylabel('Test MSE')
plt.title('Q6: Regularization — Ridge vs Lasso Test MSE vs Alpha')
plt.legend()
plt.show()

best_ridge_alpha = alphas[np.argmin(ridge_test_mse)]
best_lasso_alpha = alphas[np.argmin(lasso_test_mse)]
print(f'Best Ridge alpha: {best_ridge_alpha:.4f}')
print(f'Best Lasso alpha: {best_lasso_alpha:.4f}')

### Q7: What is cross-validation and why is it better than a single train/test split?

**Answer:** Cross-validation uses all data for both training and validation by splitting into K folds:
1. Split data into K equal folds
2. For each fold: train on K-1 folds, validate on held-out fold
3. Average the K validation scores

**Advantages:** Lower variance estimate of performance, uses all data for training.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold

X_cv, y_cv = make_classification(n_samples=300, n_features=10, n_informative=5, random_state=42)

model_cv = RandomForestClassifier(n_estimators=50, random_state=42)

# Single train/test split — high variance
single_scores = []
for seed in range(20):
    Xcv_tr, Xcv_te, ycv_tr, ycv_te = train_test_split(X_cv, y_cv, test_size=0.3, random_state=seed)
    model_cv.fit(Xcv_tr, ycv_tr)
    single_scores.append(model_cv.score(Xcv_te, ycv_te))

# 5-fold CV — lower variance
cv_scores = cross_val_score(model_cv, X_cv, y_cv, cv=5, scoring='accuracy')

print('Single 80/20 split (20 random seeds):')
print(f'  Mean: {np.mean(single_scores):.4f}  Std: {np.std(single_scores):.4f}')
print()
print('5-Fold Cross-Validation:')
print(f'  Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}')
print(f'  Fold scores: {np.round(cv_scores, 4)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(single_scores, bins=10, color='steelblue', edgecolor='k')
axes[0].set_title('Single 80/20 Split (20 seeds)\nHigh variance!')
axes[0].set_xlabel('Accuracy')
axes[1].bar(range(1, 6), cv_scores, color='darkorange', edgecolor='k')
axes[1].axhline(cv_scores.mean(), color='black', linestyle='--', lw=2, label=f'Mean={cv_scores.mean():.3f}')
axes[1].set_title('5-Fold CV Scores\nLow variance!')
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()

### Q8: Explain the bias-variance tradeoff

**Answer:** Total prediction error = Bias^2 + Variance + Irreducible Noise

- **Bias:** Error from wrong assumptions in the model (underfitting). High bias = simple model misses patterns.
- **Variance:** Error from sensitivity to training data fluctuations (overfitting). High variance = memorizes training noise.
- **Trade-off:** Reducing bias usually increases variance and vice versa. The goal is to find the sweet spot.

In [ ]:
# Bias-variance decomposition simulation
np.random.seed(42)

def true_func(x):
    return np.sin(2 * np.pi * x)

n_datasets = 50
n_train    = 20
n_test     = 100

x_test = np.linspace(0, 1, n_test)
y_true = true_func(x_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
degrees = [1, 5, 15]

for ax, degree in zip(axes, degrees):
    all_preds = []
    for _ in range(n_datasets):
        x_tr = np.sort(np.random.rand(n_train))
        y_tr = true_func(x_tr) + np.random.randn(n_train) * 0.2

        coeffs = np.polyfit(x_tr, y_tr, degree)
        poly   = np.poly1d(coeffs)
        preds  = poly(x_test)
        all_preds.append(preds)
        if _ < 10:
            ax.plot(x_test, preds, 'gray', alpha=0.2)

    all_preds = np.array(all_preds)
    mean_pred = all_preds.mean(axis=0)
    bias_sq   = np.mean((mean_pred - y_true) ** 2)
    variance  = np.mean(all_preds.var(axis=0))

    ax.plot(x_test, y_true,   'r-', lw=2, label='True function')
    ax.plot(x_test, mean_pred,'b-', lw=2, label='Mean prediction')
    ax.set_title(f'Degree {degree}\nBias^2={bias_sq:.3f}  Variance={variance:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=9)
    ax.set_ylim(-2.5, 2.5)

plt.suptitle('Q8: Bias-Variance Tradeoff — Polynomial Fitting', fontsize=13)
plt.tight_layout()
plt.show()

### Q9: How does gradient boosting work?

**Answer:** Gradient Boosting builds trees sequentially, where each tree fits the RESIDUALS (errors) of the previous trees:

1. Start with an initial prediction (e.g., mean of y)
2. Compute residuals = y_true - y_pred
3. Fit a tree to the residuals
4. Update prediction: y_pred += learning_rate * tree_prediction
5. Repeat until convergence

**Key insight:** We're doing gradient descent in FUNCTION space — each tree is a gradient step.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Show how residuals shrink with each boosting round
X_gb, y_gb = make_classification(n_samples=500, n_features=5, n_informative=3, random_state=42)
Xgb_tr, Xgb_te, ygb_tr, ygb_te = train_test_split(X_gb, y_gb, test_size=0.2, random_state=42)

n_estimators_range = [1, 5, 10, 20, 50, 100, 200]
gb_train_acc = []
gb_test_acc  = []

for n in n_estimators_range:
    gb = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1,
                                     max_depth=3, random_state=42)
    gb.fit(Xgb_tr, ygb_tr)
    gb_train_acc.append(gb.score(Xgb_tr, ygb_tr))
    gb_test_acc.append(gb.score(Xgb_te, ygb_te))

plt.figure(figsize=(10, 5))
plt.plot(n_estimators_range, gb_train_acc, 'b-o', label='Train', markersize=7)
plt.plot(n_estimators_range, gb_test_acc,  'r-o', label='Test',  markersize=7)
plt.xlabel('Number of Boosting Rounds')
plt.ylabel('Accuracy')
plt.title('Q9: Gradient Boosting — Accuracy Improves with More Rounds')
plt.legend()
plt.show()
print(f'Final test accuracy at {n_estimators_range[-1]} rounds: {gb_test_acc[-1]:.4f}')

### Q10: What is a ROC curve and what does AUC mean?

**Answer:**
- **ROC (Receiver Operating Characteristic):** Plots True Positive Rate vs False Positive Rate at every classification threshold.
- **AUC (Area Under Curve):** Single number summary. AUC=0.5 means random, AUC=1.0 means perfect.
- **When to use:** Use AUC when class distribution is imbalanced or you care about ranking, not just accuracy.

In [ ]:
from sklearn.metrics import roc_curve, auc, RocCurveDisplay

X_roc, y_roc = make_classification(n_samples=1000, n_features=10, n_informative=5,
                                    class_sep=0.8, random_state=42)
Xroc_tr, Xroc_te, yroc_tr, yroc_te = train_test_split(X_roc, y_roc, test_size=0.3, random_state=42)

models_roc = {
    'Logistic Regression': LogisticRegression(max_iter=200),
    'Decision Tree':       DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             xgb.XGBClassifier(n_estimators=100, verbosity=0,
                                              use_label_encoder=False, eval_metric='logloss', random_state=42)
}

plt.figure(figsize=(9, 7))
colors = ['steelblue', 'darkorange', 'green', 'purple']

for (name, model), color in zip(models_roc.items(), colors):
    model.fit(Xroc_tr, yroc_tr)
    proba = model.predict_proba(Xroc_te)[:, 1]
    fpr, tpr, _ = roc_curve(yroc_te, proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random (AUC=0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Q10: ROC Curves — Model Comparison')
plt.legend(loc='lower right')
plt.show()

### Q11: What is feature scaling and when is it required?

**Answer:**
- **Required for:** Linear models, SVM, KNN, PCA, neural nets — because these algorithms use distances or gradients sensitive to magnitude.
- **NOT required for:** Tree-based models (splits on value thresholds, not distances).

**Methods:**
- `StandardScaler`: zero mean, unit variance: x = (x - mean) / std
- `MinMaxScaler`: scales to [0, 1]: x = (x - min) / (max - min)
- `RobustScaler`: uses median/IQR — robust to outliers

In [ ]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# Show SVM degrading without scaling
X_sc, y_sc = make_classification(n_samples=500, n_features=5, n_informative=3, random_state=42)

# Artificially create very different feature scales
X_sc_unscaled = X_sc.copy()
X_sc_unscaled[:, 0] *= 1000  # feature 0: scale ~1000x
X_sc_unscaled[:, 1] *= 0.001 # feature 1: scale ~0.001x

Xsc_tr, Xsc_te, ysc_tr, ysc_te = train_test_split(
    X_sc_unscaled, y_sc, test_size=0.3, random_state=42
)

# Without scaling
svm_no_scale = SVC(kernel='rbf', C=1.0)
svm_no_scale.fit(Xsc_tr, ysc_tr)
acc_no_scale = svm_no_scale.score(Xsc_te, ysc_te)

# With StandardScaler
scaler_q = StandardScaler()
Xsc_tr_s = scaler_q.fit_transform(Xsc_tr)
Xsc_te_s = scaler_q.transform(Xsc_te)
svm_scaled = SVC(kernel='rbf', C=1.0)
svm_scaled.fit(Xsc_tr_s, ysc_tr)
acc_scaled = svm_scaled.score(Xsc_te_s, ysc_te)

# Random Forest - should be unaffected
rf_no_scale = RandomForestClassifier(n_estimators=100, random_state=42)
rf_no_scale.fit(Xsc_tr, ysc_tr)
acc_rf = rf_no_scale.score(Xsc_te, ysc_te)

print('SVM without scaling:  ', round(acc_no_scale, 4))
print('SVM with scaling:     ', round(acc_scaled, 4))
print('Random Forest (no scaling needed):', round(acc_rf, 4))

### Q12: What is the difference between precision, recall, and F1?

**Formulas:**
```
Precision = TP / (TP + FP)  -- when you predict positive, how often correct?
Recall    = TP / (TP + FN)  -- of all actual positives, how many found?
F1        = 2 * P * R / (P + R)  -- harmonic mean
```
**When to use each:**
- **High Precision** matters when false positives are costly (spam detection, fraud alert)
- **High Recall** matters when false negatives are costly (cancer screening, security)
- **F1** balances both — use when class imbalance exists

In [ ]:
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

# Imbalanced dataset
X_pr, y_pr = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    weights=[0.9, 0.1],  # 90% class 0, 10% class 1
    random_state=42
)
Xpr_tr, Xpr_te, ypr_tr, ypr_te = train_test_split(X_pr, y_pr, test_size=0.3, random_state=42)

rf_pr = RandomForestClassifier(n_estimators=100, random_state=42)
rf_pr.fit(Xpr_tr, ypr_tr)
proba_pr = rf_pr.predict_proba(Xpr_te)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(ypr_te, proba_pr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recalls, precisions, 'b-', lw=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve (Imbalanced Data)')
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# Confusion Matrix at default threshold
cm = confusion_matrix(ypr_te, rf_pr.predict(Xpr_te))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
disp.plot(ax=axes[1], colorbar=False)
axes[1].set_title('Confusion Matrix (threshold=0.5)')

plt.tight_layout()
plt.show()
print(classification_report(ypr_te, rf_pr.predict(Xpr_te)))

### Q13: What is PCA and when would you use it?

**Answer:** Principal Component Analysis finds orthogonal directions (principal components) of maximum variance in the data and projects data onto them.

**Use when:**
- High-dimensional data with correlated features
- Visualization (reduce to 2D/3D)
- Remove multicollinearity before linear models
- Speed up downstream ML

**Caution:** PCA components are not interpretable. Loses some information.

In [ ]:
from sklearn.decomposition import PCA

# Use make_classification with many features
X_pca, y_pca = make_classification(n_samples=500, n_features=20, n_informative=5,
                                    n_redundant=10, random_state=42)
scaler_pca = StandardScaler()
X_pca_s = scaler_pca.fit_transform(X_pca)

pca = PCA(n_components=20)
pca.fit(X_pca_s)

explained_var = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 21), pca.explained_variance_ratio_, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Scree Plot — Variance per Component')

axes[1].plot(range(1, 21), explained_var, 'b-o', markersize=6)
axes[1].axhline(0.95, color='red', linestyle='--', label='95% variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

n_for_95 = np.argmax(explained_var >= 0.95) + 1
print(f'Components needed for 95% variance: {n_for_95} out of 20')

### Q14: How do you handle class imbalance?

**Answer (multiple strategies):**
1. **Resampling:** Oversample minority (SMOTE), undersample majority, or both
2. **Class weights:** `class_weight='balanced'` in sklearn — costs misclassifying minority more
3. **Threshold tuning:** Move decision threshold from 0.5 to a lower value
4. **Evaluation:** Never use accuracy — use F1, AUC-ROC, or AUC-PR
5. **Algorithm choice:** Algorithms like XGBoost have `scale_pos_weight` parameter

In [ ]:
from sklearn.utils import resample

# Create imbalanced dataset
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    weights=[0.95, 0.05], random_state=42
)
Ximb_tr, Ximb_te, yimb_tr, yimb_te = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb
)

print(f'Class distribution (train): {np.bincount(yimb_tr)}')
print(f'Minority class %: {yimb_tr.mean()*100:.1f}%')

# Strategy 1: No rebalancing
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(Ximb_tr, yimb_tr)

# Strategy 2: class_weight balanced
rf_balanced = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_balanced.fit(Ximb_tr, yimb_tr)

# Strategy 3: Oversample minority in training set
X_maj = Ximb_tr[yimb_tr == 0]
X_min = Ximb_tr[yimb_tr == 1]
y_maj = np.zeros(len(X_maj), dtype=int)
y_min = np.ones(len(X_min), dtype=int)

X_min_up, y_min_up = resample(X_min, y_min, n_samples=len(X_maj), random_state=42)
X_os = np.vstack([X_maj, X_min_up])
y_os = np.concatenate([y_maj, y_min_up])

rf_oversample = RandomForestClassifier(n_estimators=100, random_state=42)
rf_oversample.fit(X_os, y_os)

from sklearn.metrics import f1_score
print()
print(f"{'Strategy':<25} {'Accuracy':>10} {'F1 (minority)':>15}")
print('-' * 52)
for name, mdl in [('Baseline', rf_baseline), ('Balanced weights', rf_balanced), ('Oversample', rf_oversample)]:
    pred = mdl.predict(Ximb_te)
    print(f"{name:<25} {accuracy_score(yimb_te, pred):>10.4f} {f1_score(yimb_te, pred):>15.4f}")

### Q15: What is the curse of dimensionality?

**Answer:** As the number of features grows, the volume of the feature space grows exponentially. Data becomes increasingly sparse — nearest neighbors are no longer meaningfully close.

**Effects:**
- Distance-based algorithms (KNN, SVM, K-means) degrade
- More features = more overfitting risk
- Need exponentially more data to fill high-dimensional space

**Solutions:** PCA/dimensionality reduction, feature selection, regularization

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Show KNN degrading with more dimensions
np.random.seed(42)

dims = [2, 5, 10, 20, 50, 100, 200]
knn_accs = []

for d in dims:
    X_dim, y_dim = make_classification(
        n_samples=500, n_features=d, n_informative=max(2, d//5),
        n_redundant=0, n_repeated=0, random_state=42
    )
    scaler_dim = StandardScaler()
    X_dim_s = scaler_dim.fit_transform(X_dim)
    Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_dim_s, y_dim, test_size=0.3, random_state=42)

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(Xd_tr, yd_tr)
    knn_accs.append(knn.score(Xd_te, yd_te))

plt.figure(figsize=(10, 5))
plt.plot(dims, knn_accs, 'b-o', lw=2, markersize=8)
plt.xlabel('Number of Features (Dimensions)')
plt.ylabel('KNN Test Accuracy')
plt.title('Q15: Curse of Dimensionality — KNN Accuracy vs Dimensions')
plt.xscale('log')
plt.grid(True, alpha=0.4)
plt.show()
print('Dims  | KNN Accuracy')
for d, a in zip(dims, knn_accs):
    print(f'{d:5d} | {a:.4f}')

### Q16-Q20: Rapid-Fire Interview Questions

**Q16: What is the difference between bagging and boosting?**
- **Bagging:** Train N independent models in parallel, average predictions. Reduces variance. (Random Forest)
- **Boosting:** Train models sequentially, each correcting previous errors. Reduces bias. (XGBoost, AdaBoost)

**Q17: When would you use Logistic Regression over Random Forest?**
- When interpretability is paramount (medical decisions, legal compliance)
- When dataset is small (RF can overfit on small data)
- When you need probability calibration out-of-the-box
- When training speed matters for production inference

**Q18: What is multicollinearity and why is it a problem?**
- When features are highly correlated with each other
- Makes linear model coefficients unstable and uninterpretable
- Fix: VIF analysis, PCA, L2 regularization (Ridge), drop correlated features

**Q19: What is the difference between generative and discriminative models?**
- **Discriminative:** Models P(y|X) directly — learn decision boundary (Logistic Regression, SVM, Neural Nets)
- **Generative:** Models P(X|y) and P(y), uses Bayes theorem for P(y|X) (Naive Bayes, Gaussian Mixture Models)
- Discriminative usually better when enough data; generative better with limited data or for generation

**Q20: How do you detect and handle outliers in ML?**
1. **Detect:** Box plots, z-score (>3), IQR method, Isolation Forest
2. **Handle:** Remove (if data entry error), cap/clip (winsorization), use robust algorithms (tree-based), robust scalers

In [ ]:
# Q20 Demo: Outlier detection and impact on models
np.random.seed(42)
X_out = np.random.randn(200, 2)
y_out = (X_out[:, 0] + X_out[:, 1] > 0).astype(int)

# Add outliers
X_out_noisy = X_out.copy()
X_out_noisy[0:5] = np.array([[10, 10], [-10, 10], [10, -10], [-10, -10], [8, -8]])

# Isolation Forest for detection
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=42)
outlier_preds = iso.fit_predict(X_out_noisy)

inliers  = X_out_noisy[outlier_preds ==  1]
outliers = X_out_noisy[outlier_preds == -1]

plt.figure(figsize=(9, 6))
plt.scatter(inliers[:, 0],  inliers[:, 1],  c='steelblue', s=25, alpha=0.6, label='Inliers')
plt.scatter(outliers[:, 0], outliers[:, 1], c='red', s=100, marker='X', label='Detected Outliers')
plt.title('Q20: Isolation Forest — Outlier Detection')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.show()
print(f'Detected {len(outliers)} outliers out of {len(X_out_noisy)} points')

---
## Summary — Key Takeaways for ML Interviews

### Linear Models
- Gradient descent: w -= lr * gradient. Learning rate is critical.
- Ridge (L2) shrinks weights; Lasso (L1) zeros out weights (feature selection)
- Always standardize features before linear models or SVM

### Decision Trees
- Split criterion: Gini (default) or Entropy — similar results in practice
- Max depth = primary overfitting control
- Trees are invariant to feature scaling

### Ensemble Methods
- Bagging (RF): reduces variance by averaging independent models
- Boosting (XGBoost, AdaBoost): reduces bias by correcting errors sequentially
- XGBoost usually wins on tabular data: regularization, 2nd-order gradients, native NaN handling

### SVM
- Finds maximum-margin hyperplane between classes
- C: trade-off between margin width and misclassifications
- Kernel trick: implicit high-dimensional mapping via K(x,z)
- Best for: high-dimensional sparse data, small datasets

### Interview Process Tips
1. State assumptions before implementing
2. Discuss trade-offs (not just best accuracy)
3. Know when NOT to use a model
4. Always mention hyperparameters that matter most
5. Connect math to code to intuition

In [ ]:
print('Notebook complete!')
print()
print('Sections covered:')
print('  1. Linear Models from Scratch (GD Linear Regression + Logistic Regression)')
print('  2. Decision Tree Internals (Gini, scratch implementation, overfitting)')
print('  3. Ensemble Methods (Bagging math, RF feature importance, AdaBoost weights, XGBoost)')
print('  4. SVM Margin Concept (C parameter, kernel trick, support vectors)')
print('  5. 20 Interview Questions with Code Demonstrations')
print()
print('Datasets used: California Housing, Titanic (seaborn), make_classification, make_moons, make_circles')